# Modelação — Deteção de Conexões Maliciosas (IAAC — Grupo 10)

Terceiro e último notebook do roadmap adaptado: comparação de modelos por validação cruzada, afinação do threshold de decisão e avaliação final — **uma única vez** — no conjunto de teste.

Pré-requisito: `notebooks/Data_Preparation_cybersecurity.ipynb` (split, pipeline, `High_Failed_Logins`).

Critérios de sucesso definidos no Business Understanding: **recall ≥ 90%** e **taxa de falsos positivos ≤ 1%**.


In [1]:
import pandas as pd, numpy as npfrom sklearn.model_selection import train_test_split, StratifiedKFold, cross_validatefrom sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformerfrom sklearn.compose import ColumnTransformerfrom sklearn.pipeline import Pipelinefrom sklearn.impute import SimpleImputerfrom sklearn.linear_model import LogisticRegressionfrom sklearn.ensemble import RandomForestClassifierfrom sklearn.metrics import (recall_score, precision_score, roc_auc_score,                              average_precision_score, confusion_matrix, make_scorer,                              precision_recall_curve)df = pd.read_csv("../datasets/Raw/cybersecurity_network_logs.csv")target = "Is_Malicious"num_cols_log = ["Packet_Size_Bytes", "Geo_Distance_km"]num_cols_plain = ["Connection_Duration_ms", "Failed_Logins"]cat_cols = ["Protocol"]X = df.drop(columns=[target])y = df[target]# mesmo split do notebook de Data Preparation (mesma seed)X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.30, stratify=y, random_state=42)X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=42)def add_features(df_in):    df2 = df_in.copy()    df2["High_Failed_Logins"] = (df2["Failed_Logins"] >= 3).astype(int)    return df2X_train_fe = add_features(X_train)X_val_fe   = add_features(X_val)X_test_fe  = add_features(X_test)num_cols_plain_fe = num_cols_plain + ["High_Failed_Logins"]log_pipeline = Pipeline([("impute", SimpleImputer(strategy="median")),                          ("log1p", FunctionTransformer(np.log1p, feature_names_out="one-to-one")),                          ("scale", StandardScaler())])plain_pipeline = Pipeline([("impute", SimpleImputer(strategy="median")), ("scale", StandardScaler())])cat_pipeline = Pipeline([("impute", SimpleImputer(strategy="most_frequent")), ("onehot", OneHotEncoder(handle_unknown="ignore"))])preprocessor = ColumnTransformer([    ("log_num", log_pipeline, num_cols_log),    ("plain_num", plain_pipeline, num_cols_plain_fe),    ("cat", cat_pipeline, cat_cols),])

## 1. Comparação de modelos (5-fold cross-validation, só no treino)

Dois candidatos: **Regressão Logística** (simples, interpretável) vs. **Random Forest** (não-linear, lida melhor com a relação bimodal encontrada na EDA para `Packet_Size_Bytes`/`Geo_Distance_km`). Ambos com `class_weight="balanced"`. Métrica de seleção: **PR-AUC média** (mais informativa que ROC-AUC com 4,99% de prevalência).


In [1]:
models = {    "Logistic Regression": LogisticRegression(class_weight="balanced", max_iter=2000, random_state=42),    "Random Forest": RandomForestClassifier(n_estimators=200, class_weight="balanced", random_state=42, n_jobs=-1),}cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)scoring = {"recall": make_scorer(recall_score), "precision": make_scorer(precision_score, zero_division=0),           "roc_auc": "roc_auc", "pr_auc": "average_precision"}cv_results = {}for name, clf in models.items():    pipe = Pipeline([("preprocess", preprocessor), ("model", clf)])    res = cross_validate(pipe, X_train_fe, y_train, cv=cv, scoring=scoring, n_jobs=-1)    means = {k: res[f"test_{k}"].mean() for k in scoring}    stds  = {k: res[f"test_{k}"].std() for k in scoring}    print(f"\n{name}")    for k in scoring:        print(f"  {k:10s}: {means[k]:.4f} (+/- {stds[k]:.4f})")

Logistic Regression
  recall    : 0.8591 (+/- 0.0208)
  precision : 0.5126 (+/- 0.0215)
  roc_auc   : 0.9465 (+/- 0.0049)
  pr_auc    : 0.8657 (+/- 0.0171)

Random Forest
  recall    : 0.9152 (+/- 0.0144)
  precision : 0.9761 (+/- 0.0210)
  roc_auc   : 0.9584 (+/- 0.0079)
  pr_auc    : 0.9264 (+/- 0.0115)


**Random Forest vence claramente** em todas as métricas — sobretudo em precisão (0,976 vs. 0,513) e PR-AUC (0,926 vs. 0,866). Confirma a leitura da EDA: a relação entre as features e a classe maliciosa é não-linear/bimodal, o que penaliza um modelo linear como a Regressão Logística.


## 2. Treinar o modelo escolhido no treino completo


In [1]:
best_pipe = Pipeline([("preprocess", preprocessor), ("model", models["Random Forest"])])best_pipe.fit(X_train_fe, y_train)val_proba = best_pipe.predict_proba(X_val_fe)[:,1]val_pred_default = (val_proba >= 0.5).astype(int)print("Threshold por omissão (0.5) — validação:")print("  Recall:   ", round(recall_score(y_val, val_pred_default), 4))print("  Precisão: ", round(precision_score(y_val, val_pred_default), 4))

Threshold por omissão (0.5) — validação:
  Recall:    0.893
  Precisão:  0.9766


## 3. Afinar o threshold de decisão (na validação)

Com threshold 0,5 o recall (89,3%) fica ligeiramente abaixo do alvo de 90%. Como a taxa de falsos positivos está muito longe do limite (0,11% vs. 1%), há margem para baixar o threshold: procuramos, na curva precisão-recall, o **threshold mais alto (= mais preciso) entre os que cumprem recall ≥ 90% e FP ≤ 1%**.


In [1]:
precisions, recalls, thresholds = precision_recall_curve(y_val, val_proba)candidates = []for p, r, t in zip(precisions[:-1], recalls[:-1], thresholds):    y_pred_t = (val_proba >= t).astype(int)    tn, fp, fn, tp = confusion_matrix(y_val, y_pred_t).ravel()    fp_rate = fp / (fp + tn)    if r >= 0.90 and fp_rate <= 0.01:        candidates.append((t, r, fp_rate, p))best_t, best_r, best_fp, best_p = max(candidates, key=lambda c: c[3])print(f"Threshold escolhido: {best_t:.3f}")print(f"  Recall (val):    {best_r:.4f}")print(f"  Precisão (val):  {best_p:.4f}")print(f"  Taxa de FP (val):{best_fp:.4%}")

Threshold escolhido: 0.465
  Recall (val):    0.9037
  Precisão (val):  0.9769
  Taxa de FP (val):0.1123%


Com o threshold em **0,465** (em vez de 0,5), o recall na validação sobe para **90,4%**, cumprindo o critério, e a taxa de falsos positivos mantém-se em 0,11% — bem abaixo do limite de 1%. Este threshold foi escolhido **só com dados de validação**, nunca com o teste, para não enviesar a avaliação final.


## 4. Avaliação final — conjunto de teste (uma única vez)

O conjunto de teste nunca foi usado até este ponto (nem para escolher o modelo, nem para afinar o threshold). É usado agora uma única vez, para reportar a performance final.


In [1]:
test_proba = best_pipe.predict_proba(X_test_fe)[:,1]y_test_pred = (test_proba >= best_t).astype(int)cm_test = confusion_matrix(y_test, y_test_pred)tn, fp, fn, tp = cm_test.ravel()print("=== Avaliação final no TESTE ===")print("Recall:   ", round(recall_score(y_test, y_test_pred), 4))print("Precisão: ", round(precision_score(y_test, y_test_pred), 4))print("ROC-AUC:  ", round(roc_auc_score(y_test, test_proba), 4))print("PR-AUC:   ", round(average_precision_score(y_test, test_proba), 4))print("Taxa de FP:", f"{fp/(fp+tn):.4%}")print("\nMatriz de confusão:\n", cm_test)

=== Avaliação final no TESTE ===
Recall:    0.9358
Precisão:  0.9831
ROC-AUC:   0.9784
PR-AUC:    0.9571
Taxa de FP: 0.0842%

Matriz de confusão:
 [[3560    3]
 [  12  175]]


## 5. Conclusão face aos critérios de sucesso

| Critério | Alvo | Resultado no teste | Estado |
|---|---|---|---|
| Recall | ≥ 90% | **93,6%** | Cumprido |
| Taxa de falsos positivos | ≤ 1% | **0,08%** | Cumprido com folga |
| PR-AUC | — | 0,957 | — |

De 187 conexões maliciosas no teste, o modelo deteta 175 (só 12 falsos negativos) e gera apenas 3 falsos positivos em 3.563 conexões benignas. **Os dois critérios de sucesso definidos no Business Understanding foram cumpridos.**

**Cautela a registar no relatório:** o dataset parece sintético e muito limpo (sinal isolado quase perfeito em `Failed_Logins`), por isso estes resultados são provavelmente otimistas em relação a tráfego real. Recomenda-se validar o modelo com dados de produção antes de qualquer decisão de bloqueio automático, e manter o modo "alerta apenas" (sem bloqueio automático) até essa validação.

**Modelo final:** Random Forest, threshold de decisão = 0,465, pipeline completo (`preprocessor` + modelo) pronto a exportar com `joblib.dump(best_pipe, "../models/model_final.joblib")`.
